In [1]:
import duckdb
from co2sat.utils import data_dir

In [2]:
con = duckdb.connect()

In [3]:
p = str(data_dir("processed", "dynamic_features.parquet"))

In [4]:
# Hours present per (facility, date)
completeness = con.execute(f"""
    SELECT facility_id, date, COUNT(*) AS n_hours
    FROM read_parquet('{p}')
    GROUP BY facility_id, date
""").df()

In [5]:
print(completeness["n_hours"].value_counts().sort_index(ascending=False))

n_hours
24    155172
22      1158
14      1158
Name: count, dtype: int64


In [6]:
affected = completeness[completeness["n_hours"] < 24]
print(affected.groupby("date")["n_hours"].first())
print(f"\nPlant-days with full 24h: {(completeness['n_hours'] == 24).mean():.1%}")

date
2021-04-29    22
2022-09-13    14
Name: n_hours, dtype: int64

Plant-days with full 24h: 98.5%


In [ ]:
nan_stats = (
    con.execute(f"""
    SELECT {
        ", ".join(
            f"SUM(CASE WHEN band_{b:02d} IS NULL OR isnan(band_{b:02d}) THEN 1 ELSE 0 END) AS nan_b{b:02d}"
            for b in range(1, 17)
        )
    }
    FROM read_parquet('{p}')
""")
    .df()
    .T
)
print(nan_stats)

               0
nan_b01  16615.0
nan_b02  18365.0
nan_b03  17543.0
nan_b04  18538.0
nan_b05  13628.0
nan_b06  11670.0
nan_b07  15696.0
nan_b08  16986.0
nan_b09  18149.0
nan_b10  16596.0
nan_b11  13751.0
nan_b12  12520.0
nan_b13  18218.0
nan_b14  17270.0
nan_b15  14374.0
nan_b16  15775.0


In [10]:
epa = str(data_dir("processed", "epa_daily_with_attributes.parquet"))
period_filter = """
    (date BETWEEN '2021-04-01' AND '2021-05-19') OR
    (date BETWEEN '2021-09-01' AND '2021-09-30') OR
    (date BETWEEN '2022-04-01' AND '2022-04-29') OR
    (date BETWEEN '2022-09-01' AND '2022-09-28')
"""
coverage = con.execute(f"""
    WITH labels AS (
    SELECT DISTINCT facility_id, CAST(date AS DATE) AS date
    FROM read_parquet('{epa}')
    WHERE co2_metric_tons > 0 AND gross_load_mwh > 0
    AND ({period_filter})),
    sat AS (
        SELECT facility_id, date, COUNT(*) AS n_hours
        FROM read_parquet('{p}')
        GROUP BY facility_id, date
    )
    SELECT
        COUNT(*) AS labeled_plant_days,
        SUM(CASE WHEN s.n_hours IS NOT NULL THEN 1 ELSE 0 END) AS with_satellite,
        SUM(CASE WHEN s.n_hours = 24 THEN 1 ELSE 0 END) AS with_full_24h
    FROM labels l LEFT JOIN sat s USING (facility_id, date)
""").df()
print(coverage)

   labeled_plant_days  with_satellite  with_full_24h
0               90261         90261.0        88795.0


In [11]:
per_period = con.execute(f"""
    WITH labels AS (
        SELECT DISTINCT facility_id, CAST(date AS DATE) AS date
        FROM read_parquet('{epa}')
        WHERE co2_metric_tons > 0 AND gross_load_mwh > 0
        AND ({period_filter})
    ),
    sat AS (
        SELECT facility_id, date, COUNT(*) AS n_hours
        FROM read_parquet('{p}')
        GROUP BY facility_id, date
    )
    SELECT
        CASE
            WHEN l.date BETWEEN '2021-04-01' AND '2021-05-19' THEN 'P1 2021-04/05'
            WHEN l.date BETWEEN '2021-09-01' AND '2021-09-30' THEN 'P2 2021-09'
            WHEN l.date BETWEEN '2022-04-01' AND '2022-04-29' THEN 'P3 2022-04'
            ELSE 'P4 2022-09'
        END AS period,
        COUNT(*)                                        AS labeled,
        SUM(CASE WHEN s.n_hours = 24 THEN 1 END)        AS full_24h
    FROM labels l LEFT JOIN sat s USING (facility_id, date)
    GROUP BY period ORDER BY period
""").df()

per_period["paper_samples"] = [20306, 14464, 11213, 15243]  # from paper
per_period["ratio"] = (per_period["full_24h"] / per_period["paper_samples"]).round(2)
print(per_period)

          period  labeled  full_24h  paper_samples  ratio
0  P1 2021-04/05    29525   28838.0          20306   1.42
1     P2 2021-09    21696   21696.0          14464   1.50
2     P3 2022-04    17607   17607.0          11213   1.57
3     P4 2022-09    21433   20654.0          15243   1.35


In [12]:
nan_condition = " OR ".join(f"isnan(band_{b:02d})" for b in range(1, 17))

by_plant = con.execute(f"""
    SELECT facility_id, COUNT(*) AS rows_with_any_nan
    FROM read_parquet('{p}')
    WHERE {nan_condition}
    GROUP BY facility_id ORDER BY rows_with_any_nan DESC LIMIT 15
""").df()
print(by_plant)
print(f"\nTotal rows with any NaN: {by_plant['rows_with_any_nan'].sum()} (top-15 only)")

# How many plants have any NaN at all?
n_affected = con.execute(f"""
    SELECT COUNT(DISTINCT facility_id) FROM read_parquet('{p}') WHERE {nan_condition}
""").fetchone()[0]
print(f"Plants with ≥1 NaN row: {n_affected} / 1158")

Empty DataFrame
Columns: [facility_id, rows_with_any_nan]
Index: []

Total rows with any NaN: 0 (top-15 only)
Plants with ≥1 NaN row: 0 / 1158


In [13]:
by_hour = con.execute(f"""
    SELECT date, hour, COUNT(*) AS n_plants_affected
    FROM read_parquet('{p}')
    WHERE {nan_condition}
    GROUP BY date, hour ORDER BY n_plants_affected DESC LIMIT 15
""").df()
print(by_hour)

Empty DataFrame
Columns: [date, hour, n_plants_affected]
Index: []


In [14]:
hours_0429 = (
    con.execute(f"""
    SELECT DISTINCT hour FROM read_parquet('{p}')
    WHERE date = '2021-04-29' ORDER BY hour
""")
    .df()["hour"]
    .tolist()
)
missing_0429 = sorted(set(range(24)) - set(hours_0429))
print(f"2021-04-29 missing hours: {missing_0429}")

# Same for the drop-date, for the record
hours_0913 = (
    con.execute(f"""
    SELECT DISTINCT hour FROM read_parquet('{p}')
    WHERE date = '2022-09-13' ORDER BY hour
""")
    .df()["hour"]
    .tolist()
)
print(f"2022-09-13 missing hours: {sorted(set(range(24)) - set(hours_0913))}")

2021-04-29 missing hours: [21, 22]
2022-09-13 missing hours: [11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
